In [1]:
import numpy as np  # linear algebra
import pandas as pd  # data processing, CSV file I/O (e.g. pd.read_csv)

import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))




/kaggle/input/train.csv
/kaggle/input/test.csv
/kaggle/input/description.md
/kaggle/input/sample_submission.csv
/kaggle/input/train.csv.zip
/kaggle/input/test.csv.zip
/kaggle/input/sample_submission.csv.zip
/kaggle/input/tweet-sentiment-extraction/train.csv
/kaggle/input/tweet-sentiment-extraction/test.csv
/kaggle/input/tweet-sentiment-extraction/description.md
/kaggle/input/tweet-sentiment-extraction/sample_submission.csv
/kaggle/input/tweet-sentiment-extraction/train.csv.zip
/kaggle/input/tweet-sentiment-extraction/test.csv.zip
/kaggle/input/tweet-sentiment-extraction/sample_submission.csv.zip


In [2]:
import os

os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

import numpy as np
import pandas as pd

try:
    import tokenizers
except Exception:
    tokenizers = None

import torch
import torch.nn as nn
from tqdm import tqdm
import string
import csv

try:
    import transformers
except Exception:
    transformers = None


from sklearn import model_selection
from torch.optim import AdamW


class config:
    MAX_LEN = 141
    TRAIN_BATCH_SIZE = 40  # 8 #??
    VALID_BATCH_SIZE = 16  # 4 #??
    EPOCHS = 10
    BERT_PATH = "../input/bertbaseuncased/"
    MODEL_PATH = "model.bin"
    TRAINING_FILE = "../input/tweet-sentiment-extraction/train.csv"
    TOKENIZER = None


class SimpleTokenizer:
    def __init__(self):
        pass

    def encode(self, text):
        tokens = text.split()
        ids = list(range(len(tokens)))
        offsets = []
        pos = 0
        for tok in tokens:
            start = text.find(tok, pos)
            end = start + len(tok)
            offsets.append((start, end))
            pos = end
        offsets = [(0, 0)] + offsets + [(len(text), len(text))]
        ids = [101] + ids + [102]  # dummy CLS/SEP ids
        return type(
            "EncodeResult",
            (),
            {
                "tokens": ["[CLS]"] + tokens + ["[SEP]"],
                "ids": ids,
                "offsets": offsets,
            },
        )


try:
    vocab_path = os.path.join(config.BERT_PATH, "vocab.txt")
    if os.path.isfile(vocab_path) and tokenizers is not None:
        config.TOKENIZER = tokenizers.BertWordPieceTokenizer(vocab_path, lowercase=True)
    else:
        raise FileNotFoundError
except Exception:
    try:
        if transformers is not None:
            config.TOKENIZER = transformers.AutoTokenizer.from_pretrained(
                "bert-base-uncased", use_fast=True
            )
        else:
            raise ImportError
    except Exception:
        config.TOKENIZER = SimpleTokenizer()


class BERTBaseUncased(nn.Module):
    def __init__(self):
        super(BERTBaseUncased, self).__init__()
        try:
            if transformers is not None and os.path.isdir(config.BERT_PATH):
                self.bert = transformers.BertModel.from_pretrained(config.BERT_PATH)
            else:
                raise ImportError
        except Exception as e:
            print(
                f"Warning: could not load pretrained BERT model ({e}); using zero‑tensor fallback."
            )
            self.bert = None
        self.l0 = nn.Linear(768, 2)

    def forward(self, ids, mask, token_type_ids):
        if self.bert is not None:
            sequence_output, pooled_output = self.bert(
                ids, attention_mask=mask, token_type_ids=token_type_ids
            )
        else:
            batch_size, seq_len = ids.shape
            sequence_output = torch.zeros(
                (batch_size, seq_len, 768), device=ids.device, dtype=torch.float32
            )
        logits = self.l0(sequence_output)
        start_logits, end_logits = logits.split(1, dim=-1)
        start_logits = start_logits.squeeze(-1)
        end_logits = end_logits.squeeze(-1)
        return start_logits, end_logits


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BERTBaseUncased()
model.to(device)
model = nn.DataParallel(model)

pretrained_path = "../input/tweet-sentiment-extraction-training/model.bin"
if os.path.isfile(pretrained_path):
    try:
        model.load_state_dict(torch.load(pretrained_path, map_location=device))
    except Exception:
        print("Failed to load pretrained weights – using uninitialised model.")
else:
    print("Pretrained model.bin not found – using randomly initialized model.")

model.eval()


class TweetDataset:
    """initialization function"""

    def __init__(self, tweet, sentiment, selected_text):
        self.tweet = tweet
        self.sentiment = sentiment
        self.selected_text = selected_text
        self.max_len = config.MAX_LEN
        self.tokenizer = config.TOKENIZER

    def __len__(self):
        return len(self.tweet)

    def _encode(self, text):
        """Unified tokenisation handling for different tokenizer types."""
        result = self.tokenizer.encode(text)

        if (
            hasattr(result, "tokens")
            and hasattr(result, "ids")
            and hasattr(result, "offsets")
        ):
            tokens = result.tokens
            ids = result.ids
            offsets = result.offsets
        elif isinstance(result, list):
            ids = result
            if hasattr(self.tokenizer, "convert_ids_to_tokens"):
                tokens = self.tokenizer.convert_ids_to_tokens(ids)
            else:
                tokens = ["[UNK]"] * len(ids)
            simple = SimpleTokenizer()
            simple_res = simple.encode(text)
            offsets = simple_res.offsets
        else:
            simple = SimpleTokenizer()
            simple_res = simple.encode(text)
            tokens = simple_res.tokens
            ids = simple_res.ids
            offsets = simple_res.offsets

        return tokens, ids, offsets

    def __getitem__(self, item):
        tweet = str(self.tweet[item])
        tweet = " ".join(tweet.split())

        selected_text = str(self.selected_text[item])
        selected_text = " ".join(selected_text.split())

        len_sel_text = len(selected_text)

        idx0 = -1
        idx1 = -1
        for ind in (i for i, e in enumerate(tweet) if e == selected_text[0]):
            if tweet[ind : ind + len_sel_text] == selected_text:
                idx0 = ind
                idx1 = ind + len_sel_text - 1
                break

        char_targets = [0] * len(tweet)
        if idx0 != -1 and idx1 != -1:
            for j in range(idx0, idx1 + 1):
                if tweet[j] != " ":
                    char_targets[j] = 1

        tok_tweet_tokens, tok_tweet_ids, tok_tweet_offsets = self._encode(tweet)

        tok_tweet_offsets = tok_tweet_offsets[1:-1]

        targets = [0] * (len(tok_tweet_tokens) - 2)
        for j, (offset1, offset2) in enumerate(tok_tweet_offsets):
            if sum(char_targets[offset1:offset2]) > 0:
                targets[j] = 1

        targets = [0] + targets + [0]
        targets_start = [0] * len(targets)
        targets_end = [0] * len(targets)

        non_zero = np.nonzero(targets)[0]

        if len(non_zero) > 0:
            targets_start[non_zero[0]] = 1
            targets_end[non_zero[-1]] = 1

        mask = [1] * len(tok_tweet_ids)
        token_type_ids = [0] * len(tok_tweet_ids)

        padding_len = self.max_len - len(tok_tweet_ids)
        ids = tok_tweet_ids + [0] * padding_len
        mask = mask + [0] * padding_len
        token_type_ids = token_type_ids + [0] * padding_len
        targets = targets + [0] * padding_len
        targets_start = targets_start + [0] * padding_len
        targets_end = targets_end + [0] * padding_len

        sentiment = [1, 0, 0]  # neutral
        if self.sentiment[item] == "positive":
            sentiment = [0, 0, 1]
        if self.sentiment[item] == "negative":
            sentiment = [0, 1, 0]

        return {
            "ids": torch.tensor(ids, dtype=torch.long),
            "mask": torch.tensor(mask, dtype=torch.long),
            "token_type_ids": torch.tensor(token_type_ids, dtype=torch.long),
            "targets": torch.tensor(targets, dtype=torch.long),
            "targets_start": torch.tensor(targets_start, dtype=torch.long),
            "targets_end": torch.tensor(targets_end, dtype=torch.long),
            "padding_len": torch.tensor(padding_len, dtype=torch.long),
            "tweet_tokens": " ".join(tok_tweet_tokens),
            "orig_tweet": self.tweet[item],
            "sentiment": torch.tensor(sentiment, dtype=torch.long),
            "orig_sentiment": self.sentiment[item],
            "orig_selected_text": self.selected_text[item],
        }


df_test = pd.read_csv("../input/tweet-sentiment-extraction/test.csv")
df_test.loc[:, "selected_text"] = df_test.text.values  # placeholder

test_dataset = TweetDataset(
    tweet=df_test.text.values,
    sentiment=df_test.sentiment.values,
    selected_text=df_test.selected_text.values,
)

valid_data_loader = torch.utils.data.DataLoader(
    test_dataset, shuffle=False, batch_size=config.VALID_BATCH_SIZE, num_workers=1
)

final_col = []
fin_output_start = []
fin_output_end = []
fin_padding_lens = []
fin_tweet_tokens = []
fin_orig_sentiment = []
fin_orig_selected_text = []
fin_orig_tweet = []

with torch.no_grad():
    for bi, d in enumerate(valid_data_loader):
        ids = d["ids"].to(device, dtype=torch.long)
        token_type_ids = d["token_type_ids"].to(device, dtype=torch.long)
        mask = d["mask"].to(device, dtype=torch.long)

        o1, o2 = model(ids=ids, mask=mask, token_type_ids=token_type_ids)

        fin_output_start.append(torch.sigmoid(o1).cpu().numpy())
        fin_output_end.append(torch.sigmoid(o2).cpu().numpy())
        fin_padding_lens.extend(d["padding_len"].cpu().numpy().tolist())
        fin_tweet_tokens.extend(d["tweet_tokens"])
        fin_orig_sentiment.extend(d["orig_sentiment"])
        fin_orig_selected_text.extend(d["orig_selected_text"])
        fin_orig_tweet.extend(d["orig_tweet"])

    fin_output_start = np.vstack(fin_output_start)
    fin_output_end = np.vstack(fin_output_end)

    for j in range(len(fin_tweet_tokens)):
        tweet_tokens = fin_tweet_tokens[j]
        padding_len = fin_padding_lens[j]
        original_tweet = fin_orig_tweet[j]
        sentiment = fin_orig_sentiment[j]

        seq_len = fin_output_start.shape[1] - padding_len
        start_probs = fin_output_start[j, :seq_len]
        end_probs = fin_output_end[j, :seq_len]

        # Choose start index as the highest‑probability token
        idx_start = int(np.argmax(start_probs))

        # Choose end index as the highest‑probability token *after* the start
        if idx_start < seq_len:
            idx_end = idx_start + int(np.argmax(end_probs[idx_start:]))
        else:
            idx_end = idx_start

        # Safety: ensure end is not before start
        if idx_end < idx_start:
            idx_end = idx_start

        mask = [0] * seq_len
        for mj in range(idx_start, idx_end + 1):
            mask[mj] = 1

        output_tokens = [x for p, x in enumerate(tweet_tokens.split()) if mask[p] == 1]
        output_tokens = [x for x in output_tokens if x not in ("[CLS]", "[SEP]")]

        final_output = ""
        for ot in output_tokens:
            if ot.startswith("##"):
                final_output += ot[2:]
            elif len(ot) == 1 and ot in string.punctuation:
                final_output += ot
            else:
                final_output += " " + ot
        final_output = final_output.strip()

        # If model produces empty span, fall back to the whole tweet (instead of always for neutral)
        if not final_output:
            final_output = original_tweet

        final_col.append(final_output)

submission_path = "../input/tweet-sentiment-extraction/sample_submission.csv"
if not os.path.isfile(submission_path):
    submission_path = "sample_submission.csv"  # fallback to working directory

submission = pd.read_csv(submission_path)
submission.selected_text = final_col
submission.to_csv("submission.csv", index=False)

print(submission.head())

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Pretrained model.bin not found – using randomly initialized model.
       textID                                      selected_text
0  80a1e6bc32       I just saw a shooting star... I made my wish
1  863097735d  gosh today sucks! i didnt get my tax returns! ...
2  264cd5277f  tired and didn`t really have an exciting Satur...
3  baee1e6ffc             i`ve been eating cheetos all morning..
4  67d06a8dee   haiiii sankQ i`m fineee ima js get a checkup ...
